# RSA Protocol: Computational Implementation

This notebook explores the mathematical foundations of RSA using Python in order to explore number-theoretic algorithms. The demonstrations presented will include how the Euclidean algorithm, Bézout's identity, modular inverses, modular exponentiation, and Euler's totient all serve to synthesize the creation of RSA key generation, encryption, and decryption.

Factoring using trial division and Pollard's rho algorithm will also be explored in order to show the relationship between computational complexity and the security of RSA.



## 1. Euclidean Algorithm

In order to calculate the greatest common divisor of two integers, we use the Euclidean algorithm.

Given two integers $a$ and $b$, the algorithm repeatedly computes

$$
\gcd(a,b)=\gcd(b,a\bmod b)
$$

until the remainder of zero is reached.

The Euclidean algorithm allows for the verification that the public exponent $e$ is relatively prime to Euler's totient $\phi(n)$, a necessary condition for key generation.

In [1]:
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return abs(a)


print("gcd(252, 198) =", gcd(252, 198))

assert gcd(252, 198) == 18

gcd(252, 198) = 18


## 2. Extended Euclidean Algorithm

The Extended Euclidean Algorithm creates an extension to the ordinary Euclidean Algorithm for the purpose of finding integers $x$ and $y$ satisfying Bézout's identity:

$$
ax+by=\gcd(a,b)
$$

These coefficients serve to help in calculating modular inverses, which are a necessary step for RSA private key generation.

In [2]:
def extended_gcd(a, b):
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1

    while r != 0:
        q = old_r // r

        old_r, r = r, old_r - q * r
        old_s, s = s, old_s - q * s
        old_t, t = t, old_t - q * t

    if old_r < 0:
        return -old_r, -old_s, -old_t

    return old_r, old_s, old_t


g, x, y = extended_gcd(252, 198)

print("GCD:", g)
print("Bézout coefficients:", x, y)

assert 252*x + 198*y == g

GCD: 18
Bézout coefficients: 4 -5


## 3. Modular Inverses

In order to find the modular inverse of $a$ modulo $n$, we find an integer $x$ which satisfies

$$
ax\equiv1\pmod n
$$

An inverse exists if and only if $\gcd(a,n)=1$.

The Extended Euclidean Algorithm allows for an effective method for finding the inverse.

In the context of RSA, this allows us to calculate the private exponent $d$ that satisfies

$$
ed\equiv1\pmod{\phi(n)}
$$

In [3]:
def mod_inverse(a, n):
    g, x, y = extended_gcd(a, n)

    if g != 1:
        return None

    return x % n


print("Inverse of 7 modulo 160:", mod_inverse(7, 160))

assert mod_inverse(7, 160) == 23

Inverse of 7 modulo 160: 23


## 4. Modular Exponentiation

Modular exponentiation calculates

$$
a^b\bmod n
$$

Repeated squaring allows for more efficient calculation, as opposed to merely computing the larger power.

This aspect is essential for RSA encryption and decryption because these processes require exponentiation with very large potential integers.

For a modulus that is of bit length $k$ and an exponent with a bit length of $t$, the classical implementation of repeated squaring requires $O(t)$ modular multiplications, whereas each modular multiplication costs $O(k^2)$ bit operations using schoolbook arithmetic. The combination of these two complexities gives $O(tk^2)$ total bit complexity.

In [4]:
def mod_pow(base, exponent, modulus):
    if modulus <= 0 or exponent < 0:
        raise ValueError("Invalid modulus or exponent")

    result = 1
    base = base % modulus

    while exponent > 0:
        if exponent % 2 == 1:
            result = (result * base) % modulus

        base = (base * base) % modulus
        exponent //= 2

    return result % modulus


print("3^7 mod 187 =", mod_pow(3, 7, 187))

assert mod_pow(3, 7, 187) == 130

3^7 mod 187 = 130


## 5. Euler's Totient Function

Euler's totient function $\phi(n)$ counts the number of positive integers that are less than or equal to $n$ and relatively prime to $n$.

For two unique primes $p$ and $q$,

$$
\phi(pq)=(p-1)(q-1)
$$

This provides the foundational formula for RSA key generation because the private exponent is calculated by the modular inverse of the public exponent modulo $\phi(n)$.

In [5]:
def phi_from_primes(p, q):
    return (p - 1) * (q - 1)


p = 17
q = 11

n = p * q
phi_n = phi_from_primes(p, q)

print("n =", n)
print("phi(n) =", phi_n)

assert n == 187
assert phi_n == 160

n = 187
phi(n) = 160


## 6. RSA Key Generation, Encryption, and Decryption

In order to create a public key $(e,n)$ and a private key $(d,n)$, we use two distinct prime numbers $p$ and $q$.

For the public component $e$, this must satisfy

$$
\gcd(e,\phi(n))=1
$$

Whereas the private factor satisfies

$$
ed\equiv1\pmod{\phi(n)}
$$

Encryption is executed using

$$
C=M^e\bmod n
$$

Whereas decryption recovers the original message with the equivalence

$$
M=C^d\bmod n
$$

The following implementation uses a numerical example that was used in the paper.

In [6]:
# RSA key generation
p = 17
q = 11
e = 7

n = p * q
phi_n = (p - 1) * (q - 1)

assert gcd(e, phi_n) == 1

d = mod_inverse(e, phi_n)

print("Public key:", (e, n))
print("Private key:", (d, n))

# RSA encryption
M = 3

C = mod_pow(M, e, n)

print("Original message:", M)
print("Encrypted message:", C)

# RSA decryption
decrypted = mod_pow(C, d, n)

print("Decrypted message:", decrypted)

assert decrypted == M

Public key: (7, 187)
Private key: (23, 187)
Original message: 3
Encrypted message: 130
Decrypted message: 3


## 7. Chinese Remainder Theorem

The Chinese Remainder Theorem (CRT) provides a means for solving sets of congruences when their moduli are pairwise relatively prime.

For example,

$$
x\equiv2\pmod3
$$

$$
x\equiv3\pmod5
$$

The CRT asserts that there is a unique solution modulo $15$.

In RSA, the CRT allows for RSA decryption to be computed modulo the individual prime factors $p$ and $q$, rather than directly modulo $n = pq$.

In [ ]:
def crt(a1, n1, a2, n2):
    if gcd(n1, n2) != 1:
        raise ValueError("Modulus must be relatively prime")

    inverse = mod_inverse(n1, n2)

    k = ((a2 - a1) * inverse) % n2

    return (a1 + n1 * k) % (n1 * n2)


x = crt(2, 3, 3, 5)

print("CRT solution:", x)

assert x == 8
# RSA decryption using CRT

p = 17
q = 11
n = p * q
e = 7
d = 23

M = 3
C = mod_pow(M, e, n)

# Decrypt modulo each prime
m1 = mod_pow(C, d, p)
m2 = mod_pow(C, d, q)

# Reconstruct the original message using CRT
decrypted_crt = crt(m1, p, m2, q)

print("Original message:", M)
print("Encrypted message:", C)
print("Decrypted message:", decrypted_crt)

assert decrypted_crt == M

CRT solution: 8
Original message: 3
Encrypted message: 130
Decrypted using CRT: 3


## 8. Integer Factorization by Trial Division

The goal of trial division is to factor a composite integer by iteratively checking potential divisors up to the square root.

If an integer $n$ is composite, by definition, it must have at least one nontrivial factor less than or equal to $\sqrt{n}$.

For an integer with $k$ bits, the number of trial  divisors is constrained to $O(2^{k/2})$.

This exemplifies why recovering the prime factors of a sufficiently large RSA modulus is computationally challenging using elementary factorization methods.

In [8]:
from math import isqrt

def trial_division(n):
    if n < 2:
        return None

    for i in range(2, isqrt(n) + 1):
        if n % i == 0:
            return i, n // i

    return None


print("Factors of 187:", trial_division(187))

assert trial_division(187) == (11, 17)

Factors of 187: (11, 17)


## 9. Pollard's Rho Factorization Algorithm

Pollard's rho algorithm is another means for attempting to find a nontrivial factor of a composite integer using a different method of modular arithmetic and cycle detection.

First, a sequence is generated using

$$
f(x)=x^2+c\pmod n
$$

The component of Floyd's cycle detection method compares two values which advance through a sequence at different speeds.

Under random-like behavior, a factor such as $p$ may be discovered after an approximate $O(\sqrt p)$ amount of iterations.

For an RSA modulus with primes of similar sizes, the suggested heuristic iteration complexity is approximately $O(n^{1/4})$.

This algorithm is not guaranteed to successfully produce a factor for every choice of sequence or starting value.

In [9]:
def pollards_rho(n):
    if n < 2:
        return None

    if n % 2 == 0:
        return 2

    if n % 3 == 0:
        return 3

    for c in range(1, 11):
        x = 2
        y = 2
        d = 1

        def f(x):
            return (x*x + c) % n

        for _ in range(10000):
            x = f(x)
            y = f(f(y))

            d = gcd(abs(x - y), n)

            if d == n:
                break

            if 1 < d < n:
                return d

    return None


factor = pollards_rho(187)

print("Nontrivial factor of 187:", factor)

assert factor in (11, 17)

Nontrivial factor of 187: 11


## 10. Conclusion

These implementations demonstrate how fundamental concepts from elementary number theory work to perform an effective method of RSA encryption and decryption.

Each of the concepts provides the mathematical framework necessary for efficient key generation and message encryption and decryption.

The factorization illustrations exemplified the computational difficulty associated with recovering prime factors of an RSA modulus.

Holistically, these implementations complement the accompanying expository paper by demonstrating connections between pure mathematical theory, algorithm implementation, and computational complexity.